In [ ]:
# ── 1. Repos & deps ──────────────────────────────────────────────────────────
!git clone https://github.com/goombalab/hnet.git
!git clone --recurse-submodules https://github.com/Chaoqi-LIU/oat.git
!rm -rf congenial-adventure && git clone https://github.com/SDogya/congenial-adventure.git && cp -r congenial-adventure/. . && rm -rf congenial-adventure

import os
from kaggle_secrets import UserSecretsClient
os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('wandb')

!uv add "zarr<3.0.0" dill einops numba vector-quantize-pytorch accelerate huggingface_hub robomimic torchvision wrapt pillow pandas diffusers
!uv sync

# Patch OAT's lr_scheduler.py: newer diffusers removed Union/Optional/Optimizer exports
p = 'oat/oat/model/common/lr_scheduler.py'
txt = open(p).read()
marker = 'from diffusers.optimization import ('
if marker in txt and 'from typing import Union' not in txt:
    idx = txt.index(marker)
    end_idx = txt.index(')', idx) + 1
    header = (
        'from typing import Union, Optional\n'
        'from torch.optim import Optimizer\n'
        'from diffusers.optimization import SchedulerType, TYPE_TO_SCHEDULER_FUNCTION'
    )
    open(p, 'w').write(txt[:idx] + header + txt[end_idx:])
    print('lr_scheduler.py patched OK')
else:
    print('lr_scheduler.py already clean, skipping')




In [ ]:
# ── 2. Dataset (скачиваем прямо туда, куда смотрит OAT по дефолту) ───────────
import os
from huggingface_hub import snapshot_download
from huggingface_hub import login
hf_token = UserSecretsClient().get_secret('hugface')

if hf_token:
    login(token=hf_token)
else:
    print("Ошибка: Секрет 'hugface' не найден.")


os.makedirs('/kaggle/working/oat/data/libero', exist_ok=True)
snapshot_download(
    repo_id='chaoqi-liu/libero10_N500.zarr',
    repo_type='dataset',
    local_dir='/kaggle/working/oat/data/libero'
)



# Распаковываем архив прямо в ту же папку
!unzip -q /kaggle/working/oat/data/libero/libero10_N500.zarr.zip -d /kaggle/working/oat/data/libero/

# Проверяем, что папка появилась
!ls -ld /kaggle/working/oat/data/libero/*

In [ ]:
# ── 3. Train OAT tokenizer (~2-3h, 300 epochs) ───────────────────────────────
!uv run python oat/scripts/run_workspace.py \
    --config-name=train_oattok \
    task/tokenizer=libero/libero10 \
    training.num_epochs=300 \
    logging.project=VLA-experiment \
    task.tokenizer.dataset.zarr_path="/kaggle/working/oat/data/libero/libero10_N500.zarr"

Error executing job with overrides: ['task/tokenizer=libero/libero10', 'training.num_epochs=300', 'logging.project=VLA-experiment']
Error in call to target 'oat.dataset.zarr_dataset.ZarrDataset':
TypeError('open() takes from 0 to 1 positional arguments but 2 were given')
full_key: task.tokenizer.dataset

Set the environment variable HYDRA_FULL_ERROR=1 for a complete stack trace.


In [ ]:
# ── 4. Train FD-DRAT ─────────────────────────────────────────────────────────
!TOK=$(find /kaggle/working/oat/output -name 'latest.ckpt' | tail -1) && \
 uv run run.py \
    strategy=single_gpu \
    model.tokenizer_ckpt=$TOK \
    dataset_path=/kaggle/working/oat/data/libero/libero10_N500.zarr \
    batch_size=16